In [0]:
lms      = spark.table("edtech_project.edtech_bronze.lms_events_bronze") 
sessions = spark.table("edtech_project.edtech_silver.learning_sessions")
quiz_df  = spark.read.table("edtech_project.edtech_bronze.quiz_attempts_bronze")
video_df = spark.read.table("edtech_project.edtech_bronze.video_events_bronze") 
discussion = spark.read.table("edtech_project.edtech_silver.discussion_posts_parsed")

In [0]:
from pyspark.sql.functions import col, trim, lower

def clean_keys(df):
    return df.withColumn("student_id", trim(lower(col("student_id")))) \
             .withColumn("course_id", trim(lower(col("course_id"))))

lms = clean_keys(lms)
quiz_df = clean_keys(quiz_df)
video_df = clean_keys(video_df)
discussion = clean_keys(discussion)

In [0]:
from pyspark.sql.functions import sum, avg, round

video_features = video_df.withColumn(
    "video_minutes_watched", col("watched_seconds") / 60
).groupBy("student_id", "course_id").agg(
    round(sum("video_minutes_watched"), 2).alias("total_video_minutes_watched"),
    round(avg("completion_pct"), 2).alias("video_completion_rate")
)

In [0]:
from pyspark.sql.functions import max, when

quiz_level = quiz_df.groupBy(
    "student_id", "course_id", "quiz_id"
).agg(
    max(when(col("status") == "PASSED", 1).otherwise(0)).alias("quiz_passed"),
    max("attempt_number").alias("attempts_per_quiz")
)

quiz_features = quiz_level.groupBy(
    "student_id", "course_id"
).agg(
    sum("attempts_per_quiz").alias("quiz_attempts_count"),
    round(avg("quiz_passed") * 100, 2).alias("quiz_pass_rate")
)

In [0]:
discussion_features = discussion.groupBy(
    "student_id", "course_id"
).agg(
    sum("student_posts").alias("discussion_posts_count")
)

In [0]:
from pyspark.sql.functions import current_date, datediff

login_features = lms.filter(col("event_type") == "login").groupBy(
    "student_id", "course_id"
).agg(
    sum(when(datediff(current_date(), col("event_ts")) <= 7, 1).otherwise(0)).alias("login_count_7d"),
    sum(when(datediff(current_date(), col("event_ts")) <= 30, 1).otherwise(0)).alias("login_count_30d")
)

In [0]:
from pyspark.sql import functions as F

last_active_df = last_df.groupBy("student_id", "course_id").agg(
    F.max("last_accessed_date").alias("last_active_date")
)

last_active_df = last_active_df.withColumn(
    "days_since_last_active",
    datediff(current_date(), col("last_active_date"))
)

In [0]:
from pyspark.sql.functions import weekofyear

weekly_counts = lms.groupBy(
    "student_id", "course_id", weekofyear("event_ts").alias("week")
).count()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, when

window_spec = Window.partitionBy("student_id", "course_id").orderBy("week")

weekly_counts = weekly_counts.withColumn(
    "prev_week_count",
    lag("count").over(window_spec)
)

weekly_counts = weekly_counts.withColumn(
    "engagement_trend",
    when(col("count") > col("prev_week_count"), "IMPROVING")
    .when(col("count") < col("prev_week_count"), "DECLINING")
    .otherwise("STABLE")
)

In [0]:
from pyspark.sql.functions import max as spark_max

trend_features = weekly_counts.groupBy("student_id", "course_id").agg(
    spark_max("engagement_trend").alias("engagement_trend")
)

In [0]:
base_df = video_features.select("student_id", "course_id") \
    .union(quiz_features.select("student_id", "course_id")) \
    .union(discussion_features.select("student_id", "course_id")) \
    .union(login_features.select("student_id", "course_id")) \
    .union(last_active_df.select("student_id", "course_id")) \
    .distinct()

In [0]:
final_df = base_df \
    .join(video_features, ["student_id", "course_id"], "left") \
    .join(quiz_features, ["student_id", "course_id"], "left") \
    .join(discussion_features, ["student_id", "course_id"], "left") \
    .join(login_features, ["student_id", "course_id"], "left") \
    .join(last_active_df, ["student_id", "course_id"], "left") \
    .join(trend_features, ["student_id", "course_id"], "left")

In [0]:
final_df = final_df.fillna({
    "total_video_minutes_watched": 0,
    "video_completion_rate": 0,
    "quiz_pass_rate": 0,
    "quiz_attempts_count": 0,
    "discussion_posts_count": 0,
    "login_count_7d": 0,
    "login_count_30d": 0,
    "days_since_last_active": 999,
    "engagement_trend": "STABLE"
})

In [0]:
final_df.write.mode("overwrite").saveAsTable(
    "edtech_project.edtech_silver.student_course_engagement"
)